1. Logistic Regression
- 선형회귀 방식을 분류에 적용한 알고리즘, 데이터가 어떤 범주에 속할 확률을 0~1 사이의 값으로 예측, 더 높은 범주에 속하는 쪽으로 분류
- 범주 : True/False, Yes/No, 합격/불합격
- 예: 스팸메일, 은행 대출 여부, 악성 여부, 고객의 제품 구매의사..

In [2]:
# 그래프 관련 import
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
# matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus']=False

# 판다스 관련 import
import pandas as pd
import numpy as np

# 인공지능 import
from sklearn import svm,metrics # 분류와 숫자 예측에 사용하는 모델 모음,모델의 예측 성능을 평가하는 도구 모음
from sklearn.neighbors import KNeighborsClassifier  #분류(가까운 이웃 데이터를 참고해 종류를 분류)
from sklearn.neighbors import KNeighborsRegressor   #예측(가까운 이웃데이터를 참고해 숫자를 예측)

from sklearn.preprocessing import PolynomialFeatures # 다항특성 (x², x₁ × x₂ 같은 특성을 추가)
from sklearn.preprocessing import OneHotEncoder      # 원핫인코딩(문자형 번주를 0과1로 변환)

from sklearn.linear_model import LinearRegression   #예측 (입력과 결과의 선형관례를 학습해 숫자를 예측)
from sklearn.linear_model import Ridge              #릿지 (계수가 지나치게 커지는 것을 억제하는 선형 회귀)
from sklearn.linear_model import Lasso              #라쏘 (일부 계수를 0으로 만들어 특성 선택에도 활용하는 선형 회귀)
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import mean_absolute_error # 평균절대값오차 오차범위 (실제값과 예측값의 차이를 절댓값으로 구한 뒤 평균 계산)
from sklearn.preprocessing import StandardScaler # 표준화 (각 특성을 평균 0, 표준편차 1 기준으로 표준화)
from sklearn.model_selection import train_test_split #(데이터를 학습용과 평가용으로 분리)
from scipy.special import expit                     #exip:이진분류함수
from scipy.special import softmax                   #softmax: 다중분류함수

In [5]:
# 데이터 연결
fish = pd.read_csv('data/fish_logistic.csv')
fish.head(2)
# fish.info()
# fish.describe()
fish.shape

(159, 6)

In [8]:
# 컬럼 : Species	Weight	Length	Diagonal	Height	Width
# 생선 종류: ['Bream', 'Roach', 'Whitefish', 'Parkki', 'Perch', 'Pike', 'Smelt'], Length: 7, dtype: str

In [9]:
# 데이터 분리
data= fish[fish.columns[1:]].values
lable= fish['Species'].values
data[:2]

array([[242.    ,  25.4   ,  30.    ,  11.52  ,   4.02  ],
       [290.    ,  26.3   ,  31.2   ,  12.48  ,   4.3056]])

In [10]:
# 데이터 분리
train_data, test_data, train_label, test_label = train_test_split(data, lable, random_state=42)

In [11]:
# 정규화 작업 필수
ss=StandardScaler()
ss.fit(train_data)

train_scaler = ss.transform(train_data)
test_scaler = ss.transform(test_data)

In [ ]:
# Logistic Regression
# 데이터 학습 - 학률의 형태로 보여주고 싶음.
lr= LogisticRegression(C=20, max_iter=1000)
lr.fit(train_scaler, train_label)
# 예측
predict = lr.predict(test_scaler)

# 정확도
print("train정확도:", lr.score(train_scaler, train_label))
print("test정확도:", lr.score(test_scaler, test_label))

# test데이터가 더 확률이 높음 -> 규제적용/ C규제를 적용하여 정확도 변경 가능

train정확도: 0.9327731092436975
test정확도: 0.925


In [18]:
# 예측 : 이것들이 몇퍼센트로 분리되었나?
predict = lr.predict(test_scaler[:5])
predict

array(['Perch', 'Smelt', 'Pike', 'Roach', 'Perch'], dtype=object)

In [25]:
# .decision_function(...): 각 데이터가 각 클래스에 얼마나 해당하는지 점수를 계산
# 예측 확률을 보여줌. 하지만 -도 나오면서 보여주기 어려움
decision = lr.decision_function(test_scaler[:5])
decision
print(lr.classes_) # 확률 순서(점수와 확률의 열에 대응하는 클래스 순서)

# softmax : 점수들을 0과 1 사이의 값으로 바꾸고, 합이 1이 되도록 만듬
# 7.15677789e-06 : e-06은 10의 -6제곱을 곱한다(소수점을 왼쪽으로 6칸 옮긴다)
soft1= softmax(decision, axis=1) # 행으로 보여줌
soft1

# np.round(): 숫자를 반올림하는 함수
# soft1: 반올림할 값 / decimals=3: 소수점 셋째 자리까지 남기도록 반올림
np.round(soft1, decimals=3)

['Bream' 'Parkki' 'Perch' 'Pike' 'Roach' 'Smelt' 'Whitefish']


array([[0.   , 0.014, 0.842, 0.   , 0.135, 0.007, 0.003],
       [0.   , 0.003, 0.044, 0.   , 0.007, 0.946, 0.   ],
       [0.   , 0.   , 0.034, 0.934, 0.015, 0.016, 0.   ],
       [0.011, 0.034, 0.305, 0.006, 0.567, 0.   , 0.076],
       [0.   , 0.   , 0.904, 0.002, 0.089, 0.002, 0.001]])